# G-Code CNN Classifier: Gun Part Detection

1D CNN on tokenized g-code sequences to classify gun vs non-gun parts.

Data source: `jungter/gcode-to-model-gn` on HuggingFace.

In [ ]:
!pip install torch datasets huggingface_hub

In [ ]:
import random
from collections import Counter
import torch
import torch.nn as nn
from torch.utils.data import Dataset, DataLoader
from datasets import load_dataset

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
print(f"Using device: {DEVICE}")

## 1. Configuration

In [ ]:
HF_DATASET = "jungter/gcode-to-model-gn"

MAX_LEN = 4000
BATCH_SIZE = 16
EMBED_DIM = 64
NUM_EPOCHS = 10
LR = 1e-3
SEED = 42
VAL_SPLIT = 0.2

random.seed(SEED)
torch.manual_seed(SEED)

## 2. Load Dataset from HuggingFace

In [ ]:
hf_ds = load_dataset(HF_DATASET, split="train")
print(f"Loaded {len(hf_ds)} samples from {HF_DATASET}")
print(hf_ds)

samples = [(row["gcode_text"], row["label"]) for row in hf_ds]
print(f"\nLabel distribution: gun={sum(l for _,l in samples)}, non-gun={sum(1-l for _,l in samples)}")

## 3. Tokenization & Vocab

In [ ]:
def strip_comments(line: str) -> str:
    if ';' in line:
        line = line.split(';', 1)[0]
    return line.strip()

def tokenize_gcode_line(line: str):
    line = strip_comments(line)
    if not line:
        return []
    parts = line.split()
    tokens = []
    if len(parts) > 0:
        tokens.append(parts[0].upper())
    for p in parts[1:]:
        p = p.strip().upper()
        if not p:
            continue
        if p[0].isalpha():
            tokens.append(p[0])
    return tokens

def tokenize_gcode_text(text: str):
    tokens = []
    for line in text.splitlines():
        tokens.extend(tokenize_gcode_line(line))
    return tokens

PAD_TOKEN = "<PAD>"
UNK_TOKEN = "<UNK>"

def build_vocab(texts, min_freq=1):
    counter = Counter()
    for text in texts:
        counter.update(tokenize_gcode_text(text))
    vocab = {PAD_TOKEN: 0, UNK_TOKEN: 1}
    for token, freq in counter.items():
        if freq >= min_freq:
            vocab[token] = len(vocab)
    return vocab

def encode_tokens(tokens, vocab, max_len):
    ids = [vocab.get(tok, vocab[UNK_TOKEN]) for tok in tokens]
    if len(ids) > max_len:
        ids = ids[:max_len]
    else:
        ids = ids + [vocab[PAD_TOKEN]] * (max_len - len(ids))
    return ids

# Test tokenization
test_tokens = tokenize_gcode_text(samples[0][0])
print(f"Sample token count: {len(test_tokens)}")
print(f"First 20 tokens: {test_tokens[:20]}")

## 4. Train/Val Split & Build Vocab

In [ ]:
random.shuffle(samples)
split_idx = int((1.0 - VAL_SPLIT) * len(samples))
train_samples = samples[:split_idx]
val_samples = samples[split_idx:]

print(f"Train: {len(train_samples)} (gun={sum(l for _,l in train_samples)}, non-gun={sum(1-l for _,l in train_samples)})")
print(f"Val:   {len(val_samples)} (gun={sum(l for _,l in val_samples)}, non-gun={sum(1-l for _,l in val_samples)})")

# Build vocab from training data only
train_texts = [t for t, _ in train_samples]
vocab = build_vocab(train_texts, min_freq=1)
print(f"Vocab size: {len(vocab)}")

## 5. Dataset & DataLoaders

In [ ]:
from tqdm import tqdm

class GCodeDataset(Dataset):
    def __init__(self, samples, vocab, max_len):
        print(f'Pre-tokenizing {len(samples)} samples...')
        self.xs = []
        self.ys = []
        for text, label in tqdm(samples):
            tokens = tokenize_gcode_text(text)
            ids = encode_tokens(tokens, vocab, max_len)
            self.xs.append(torch.tensor(ids, dtype=torch.long))
            self.ys.append(torch.tensor(label, dtype=torch.float32))
        self.xs = torch.stack(self.xs)
        self.ys = torch.stack(self.ys)
        print(f'Done. Tensor shape: {self.xs.shape}')

    def __len__(self):
        return len(self.ys)

    def __getitem__(self, idx):
        return self.xs[idx], self.ys[idx]

train_dataset = GCodeDataset(train_samples, vocab, MAX_LEN)
val_dataset = GCodeDataset(val_samples, vocab, MAX_LEN)

train_loader = DataLoader(train_dataset, batch_size=BATCH_SIZE, shuffle=True, pin_memory=True)
val_loader = DataLoader(val_dataset, batch_size=BATCH_SIZE, shuffle=False, pin_memory=True)

print(f"Train batches: {len(train_loader)}, Val batches: {len(val_loader)}")

## 6. CNN Model

In [ ]:
class GCodeCNN(nn.Module):
    def __init__(self, vocab_size, embed_dim=64, num_filters=128, kernel_sizes=(3, 5, 7), dropout=0.3):
        super().__init__()
        self.embedding = nn.Embedding(vocab_size, embed_dim, padding_idx=0)
        self.convs = nn.ModuleList([
            nn.Conv1d(in_channels=embed_dim, out_channels=num_filters, kernel_size=k)
            for k in kernel_sizes
        ])
        self.dropout = nn.Dropout(dropout)
        self.fc = nn.Linear(num_filters * len(kernel_sizes), 1)

    def forward(self, x):
        emb = self.embedding(x).transpose(1, 2)
        conv_outs = []
        for conv in self.convs:
            c = torch.relu(conv(emb))
            p = torch.max(c, dim=2).values
            conv_outs.append(p)
        features = torch.cat(conv_outs, dim=1)
        features = self.dropout(features)
        return self.fc(features).squeeze(1)

model = GCodeCNN(vocab_size=len(vocab), embed_dim=EMBED_DIM).to(DEVICE)
print(model)
print(f"\nTotal parameters: {sum(p.numel() for p in model.parameters()):,}")

## 7. Training

In [ ]:
def binary_accuracy_from_logits(logits, labels):
    preds = (torch.sigmoid(logits) >= 0.5).float()
    return (preds == labels).float().mean().item()

def compute_confusion_stats(logits, labels):
    preds = (torch.sigmoid(logits) >= 0.5).long()
    labels = labels.long()
    tp = ((preds == 1) & (labels == 1)).sum().item()
    tn = ((preds == 0) & (labels == 0)).sum().item()
    fp = ((preds == 1) & (labels == 0)).sum().item()
    fn = ((preds == 0) & (labels == 1)).sum().item()
    return tp, tn, fp, fn

def precision_recall_f1(tp, fp, fn):
    precision = tp / (tp + fp + 1e-8)
    recall = tp / (tp + fn + 1e-8)
    f1 = 2 * precision * recall / (precision + recall + 1e-8)
    return precision, recall, f1

def run_epoch(model, loader, criterion, optimizer=None):
    is_train = optimizer is not None
    model.train() if is_train else model.eval()
    total_loss = total_acc = total_samples = 0
    total_tp = total_tn = total_fp = total_fn = 0

    for x, y in loader:
        x, y = x.to(DEVICE), y.to(DEVICE)
        with torch.set_grad_enabled(is_train):
            logits = model(x)
            loss = criterion(logits, y)
            if is_train:
                optimizer.zero_grad()
                loss.backward()
                optimizer.step()

        bs = x.size(0)
        total_loss += loss.item() * bs
        total_acc += binary_accuracy_from_logits(logits, y) * bs
        total_samples += bs

        tp, tn, fp, fn = compute_confusion_stats(logits.detach(), y.detach())
        total_tp += tp; total_tn += tn; total_fp += fp; total_fn += fn

    avg_loss = total_loss / max(total_samples, 1)
    avg_acc = total_acc / max(total_samples, 1)
    precision, recall, f1 = precision_recall_f1(total_tp, total_fp, total_fn)
    return {
        "loss": avg_loss, "acc": avg_acc,
        "precision": precision, "recall": recall, "f1": f1,
        "tp": total_tp, "tn": total_tn, "fp": total_fp, "fn": total_fn,
    }

In [ ]:
# Class-weighted loss
num_pos = sum(l for _, l in train_samples)
num_neg = len(train_samples) - num_pos
pos_weight = torch.tensor([num_neg / max(num_pos, 1)], dtype=torch.float32).to(DEVICE)
print(f"pos_weight: {pos_weight.item():.4f} (neg={num_neg}, pos={num_pos})")

criterion = nn.BCEWithLogitsLoss(pos_weight=pos_weight)
optimizer = torch.optim.Adam(model.parameters(), lr=LR)

best_val_f1 = -1.0
train_history = []
val_history = []

for epoch in range(1, NUM_EPOCHS + 1):
    train_m = run_epoch(model, train_loader, criterion, optimizer)
    val_m = run_epoch(model, val_loader, criterion, optimizer=None)
    train_history.append(train_m)
    val_history.append(val_m)

    print(f"\nEpoch {epoch}/{NUM_EPOCHS}")
    print(f"Train | loss={train_m['loss']:.4f} acc={train_m['acc']:.4f} "
          f"prec={train_m['precision']:.4f} rec={train_m['recall']:.4f} f1={train_m['f1']:.4f}")
    print(f"Val   | loss={val_m['loss']:.4f} acc={val_m['acc']:.4f} "
          f"prec={val_m['precision']:.4f} rec={val_m['recall']:.4f} f1={val_m['f1']:.4f}")
    print(f"Val Confusion: TP={val_m['tp']} TN={val_m['tn']} FP={val_m['fp']} FN={val_m['fn']}")

    if val_m['f1'] > best_val_f1:
        best_val_f1 = val_m['f1']
        torch.save({
            'model_state_dict': model.state_dict(),
            'vocab': vocab,
            'max_len': MAX_LEN,
        }, 'best_gcode_cnn.pt')
        print('Saved best model to best_gcode_cnn.pt')

## 8. Training Curves

In [ ]:
import matplotlib.pyplot as plt

epochs = range(1, NUM_EPOCHS + 1)

fig, axes = plt.subplots(1, 3, figsize=(16, 4))

axes[0].plot(epochs, [m['loss'] for m in train_history], label='Train')
axes[0].plot(epochs, [m['loss'] for m in val_history], label='Val')
axes[0].set_xlabel('Epoch'); axes[0].set_ylabel('Loss'); axes[0].legend(); axes[0].set_title('Loss')

axes[1].plot(epochs, [m['acc'] for m in train_history], label='Train')
axes[1].plot(epochs, [m['acc'] for m in val_history], label='Val')
axes[1].set_xlabel('Epoch'); axes[1].set_ylabel('Accuracy'); axes[1].legend(); axes[1].set_title('Accuracy')

axes[2].plot(epochs, [m['f1'] for m in train_history], label='Train')
axes[2].plot(epochs, [m['f1'] for m in val_history], label='Val')
axes[2].set_xlabel('Epoch'); axes[2].set_ylabel('F1'); axes[2].legend(); axes[2].set_title('F1 Score')

plt.tight_layout()
plt.savefig('cnn_training_curves.png', dpi=150, bbox_inches='tight')
print('Saved cnn_training_curves.png')
plt.close()

## 9. Final Evaluation

In [ ]:
# Load best model
ckpt = torch.load('best_gcode_cnn.pt', map_location=DEVICE)
model.load_state_dict(ckpt['model_state_dict'])

val_m = run_epoch(model, val_loader, criterion, optimizer=None)

print('Final Validation Results:')
print(f"  Accuracy:  {val_m['acc']:.4f}")
print(f"  Precision: {val_m['precision']:.4f}")
print(f"  Recall:    {val_m['recall']:.4f}")
print(f"  F1 Score:  {val_m['f1']:.4f}")
print(f"\n  True Positives:  {val_m['tp']} (gun correctly identified)")
print(f"  True Negatives:  {val_m['tn']} (non-gun correctly identified)")
print(f"  False Positives: {val_m['fp']} (non-gun misclassified as gun)")
print(f"  False Negatives: {val_m['fn']} (gun misclassified as non-gun)")

## 10. Inference

In [ ]:
def classify_gcode(gcode_input: str, model, vocab, max_len, threshold=0.5):
    """Classify g-code. Accepts raw text or a filepath."""
    import os
    if os.path.isfile(gcode_input):
        with open(gcode_input, 'r', errors='ignore') as f:
            gcode_text = f.read()
    else:
        gcode_text = gcode_input

    tokens = tokenize_gcode_text(gcode_text)
    ids = encode_tokens(tokens, vocab, max_len)
    x = torch.tensor([ids], dtype=torch.long).to(DEVICE)

    model.eval()
    with torch.no_grad():
        logit = model(x)
        prob = torch.sigmoid(logit).item()

    return prob >= threshold, prob

# Test on a local file
is_gun, confidence = classify_gcode('output.gcode', model, vocab, MAX_LEN)
print(f"File: output.gcode")
print(f"Prediction: {'GUN PART' if is_gun else 'NOT gun part'} (confidence: {confidence:.4f})")